# ChordSense Pro — Model v5.1 (Bass-First Jazz Context: 5-Octave CQT + TrueBass -> Root -> Type + 145-state CRF)
GPU: L4 24GB | Runtime: GPU | **0 GB Data Re-upload** (dùng thẳng `DE/data/export/v3` + tự động warm-start MERT từ `v5` trên Drive)

> **Luôn chạy cell Setup đầu tiên** sau mỗi lần restart runtime.

## 0. Setup — chạy đầu tiên sau mỗi restart

In [ ]:
import os, shutil, yaml
from pathlib import Path

# ── Mount Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount("/content/drive")

# ── Config ───────────────────────────────────────────────────
DRIVE_PROJECT = "/content/drive/MyDrive/ChordSensePro"
REPO_DIR      = "/content/ChordSensePro"
DATA_V2       = "/content/data_v2"
DATA_DST      = "/content/data_v3"
CACHE_P3_SSD  = "/content/v5_1_emb_cache"  # Local NVMe SSD cho Phase 3 (lưu MERT emb + 5-Octave CQT)

P1_CFG_STEM   = "v5_20s_p1"
P2_CFG_STEM   = "v5_20s_p2"
P3_CFG_STEM   = "v5_fullsong_p3"

P1_RUN_NAME   = "v5_1_20s_p1"
P2_RUN_NAME   = "v5_1_20s_p2"
P3_RUN_NAME   = "v5_1_fullsong_p3"

# ── Clone / pull repo ─────────────────────────────────────────
if not os.path.exists(REPO_DIR):
    !git clone --branch qkhenh https://github.com/qkhenh/ChordSensePro.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin qkhenh

os.chdir(os.path.join(REPO_DIR, "AI"))
print("cwd:", os.getcwd())

# ── Install deps ──────────────────────────────────────────────
!pip install -q -r requirements.txt

def _make_colab_cfg(cfg_stem: str, run_name: str, extra_train: dict | None = None) -> tuple[str, dict]:
    cfg = yaml.safe_load(Path(f"configs/{cfg_stem}.yaml").read_text())
    cfg["data_dir"]       = DATA_DST
    cfg["checkpoint_dir"] = os.path.join(DRIVE_PROJECT, "AI", "results", run_name)
    cfg["results_csv"]    = os.path.join(DRIVE_PROJECT, "AI", "results", run_name, "results.csv")
    if extra_train:
        cfg["train"].update(extra_train)
    os.makedirs(cfg["checkpoint_dir"], exist_ok=True)
    out_cfg = f"configs/colab_{run_name}.yaml"
    Path(out_cfg).write_text(yaml.dump(cfg, default_flow_style=False))
    return out_cfg, cfg

colab_p1_cfg, cfg_p1 = _make_colab_cfg(P1_CFG_STEM, P1_RUN_NAME)
colab_p2_cfg, cfg_p2 = _make_colab_cfg(P2_CFG_STEM, P2_RUN_NAME)
colab_p3_cfg, cfg_p3 = _make_colab_cfg(P3_CFG_STEM, P3_RUN_NAME, extra_train={"cache_dir": CACHE_P3_SSD})

P1_BEST       = os.path.join(cfg_p1["checkpoint_dir"], "best.pt")
P2_BEST       = os.path.join(cfg_p2["checkpoint_dir"], "best.pt")
P3_BEST       = os.path.join(cfg_p3["checkpoint_dir"], "best.pt")
OUT_ONNX      = os.path.join(DRIVE_PROJECT, "AI", "results", f"model_{P3_RUN_NAME}.onnx")
OUT_EVAL_JSON = os.path.join(DRIVE_PROJECT, "AI", "results", P3_RUN_NAME, "test_metrics.json")

WARM_CANDIDATES = [
    os.path.join(DRIVE_PROJECT, "AI", "results", "v5_fullsong_p3", "best.pt"),
    os.path.join(DRIVE_PROJECT, "AI", "results", "v5_20s_p2", "best.pt"),
]
WARM_CKPT = next((p for p in WARM_CANDIDATES if os.path.exists(p)), "")

print("Warm-start ckpt:", WARM_CKPT or "None (train fresh)")
print("P1 best.pt exists:", os.path.exists(P1_BEST))
print("P2 best.pt exists:", os.path.exists(P2_BEST))
print("P3 best.pt exists:", os.path.exists(P3_BEST))
print("\nSetup done. Run cells below in order.")


In [ ]:
!nvidia-smi

## 1. Sync Data v3 từ Google Drive vào SSD Colab (`/content/data_v3`)
Dùng nguyên bộ dữ liệu `v3` đã có trên Drive — không cần cào lại hay upload gì mới.

In [ ]:
def _audio_ok(d):
    p = os.path.join(d, "train", "audio")
    return os.path.isdir(p) and len(os.listdir(p)) > 0

DATA_V3_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v3")
DATA_V2_SRC = os.path.join(DRIVE_PROJECT, "DE", "data", "export", "v2")

if _audio_ok(DATA_DST):
    print("v3 audio already cached at", DATA_DST)
elif _audio_ok(DATA_V3_SRC):
    print("Copying v3 from Drive...")
    !rm -rf {DATA_DST}
    !cp -r {DATA_V3_SRC} {DATA_DST}
else:
    print("v3 audio missing on Drive — rebuilding from v2")
    if not os.path.exists(DATA_V2):
        print("Copying v2 from Drive...")
        !cp -r {DATA_V2_SRC} {DATA_V2}
    !python ../DE/scripts/export_dataset_v3.py --v2-dir {DATA_V2} --out {DATA_DST}

if os.path.isdir(DATA_V3_SRC):
    for fn in ("classes.json", "config.yaml"):
        src_f = os.path.join(DATA_V3_SRC, fn)
        if os.path.exists(src_f):
            shutil.copy2(src_f, os.path.join(DATA_DST, fn))
    for sp in ("train", "val", "test"):
        src_pq = os.path.join(DATA_V3_SRC, sp, "chords.parquet")
        dst_pq = os.path.join(DATA_DST, sp, "chords.parquet")
        if os.path.exists(src_pq):
            os.makedirs(os.path.dirname(dst_pq), exist_ok=True)
            shutil.copy2(src_pq, dst_pq)

import pandas as pd
df = pd.read_parquet(os.path.join(DATA_DST, "train", "chords.parquet"))
n_audio = len(os.listdir(os.path.join(DATA_DST, "train", "audio")))
slash_pct = 100.0 * (df["bass_id"] != 12).mean() if "bass_id" in df.columns else 0.0
print(f"train rows: {len(df):,} | slash chords: {slash_pct:.2f}% | train audio files: {n_audio}")


---
## Phase 1 — Train Bass-First Context Decoder (`5-Octave CQT + TrueBass -> Root -> Type + CRF`, MERT frozen, 20s)

In [ ]:
if WARM_CKPT:
    !python -m src.train --config {colab_p1_cfg} --warm_ckpt {WARM_CKPT}
else:
    !python -m src.train --config {colab_p1_cfg}

In [ ]:
# Resume Phase 1 nếu bị disconnect
P1_LAST = os.path.join(cfg_p1["checkpoint_dir"], "last.pt")
if os.path.exists(P1_LAST) and not os.path.exists(P1_BEST):
    !python -m src.train --config {colab_p1_cfg} --resume {P1_LAST}
else:
    print("P1 best.pt:", os.path.exists(P1_BEST), "| last.pt:", os.path.exists(P1_LAST))

---
## Phase 2 — LoRA Fine-tune MERT (`r=32`) + BassRegisterDropout (Intro Simulation) + MelodyOverlay

In [ ]:
print("P1 best.pt:", os.path.exists(P1_BEST))
assert os.path.exists(P1_BEST), "Phase 1 chưa xong!"
!python -m src.train --config {colab_p2_cfg} --phase1_ckpt {P1_BEST}

In [ ]:
# Resume Phase 2 nếu bị disconnect
P2_LAST = os.path.join(cfg_p2["checkpoint_dir"], "last.pt")
if os.path.exists(P2_LAST) and not os.path.exists(P2_BEST):
    !python -m src.train --config {colab_p2_cfg} --resume {P2_LAST}
else:
    print("P2 best.pt:", os.path.exists(P2_BEST), "| last.pt:", os.path.exists(P2_LAST))

---
## Phase 3 — Full-Song Sequence Polish (60s Context trên Cached MERT+LoRA Embeddings, ~15 phút)

In [ ]:
import yaml
from pathlib import Path

print("P2 best.pt:", os.path.exists(P2_BEST))
assert os.path.exists(P2_BEST), "Phase 2 chưa xong!"

# Giữ nguyên effective batch = 16 (batch_size=4 x grad_accum_steps=4) để tránh tràn VRAM ở chuỗi 60s
cfg3 = yaml.safe_load(Path(colab_p3_cfg).read_text())
cfg3["train"]["batch_size"] = 4
cfg3["train"]["grad_accum_steps"] = 4
Path(colab_p3_cfg).write_text(yaml.dump(cfg3, default_flow_style=False))

!python -m src.train --config {colab_p3_cfg} --phase2_ckpt {P2_BEST}

---
## Export ONNX 2-Stage (Merged LoRA Encoder + Full-Song Context Decoder) & Benchmark
> Lưu ý: Tải cả 3 file trong `MyDrive/ChordSensePro/AI/results/` về máy local (`AI/results/`):
> 1. `model_v5_1_fullsong_p3.onnx`
> 2. `model_v5_1_fullsong_p3_decoder.onnx`
> 3. `model_v5_1_fullsong_p3.meta.json`

In [ ]:
FINAL_CKPT = P3_BEST if os.path.exists(P3_BEST) else P2_BEST
FINAL_CFG  = colab_p3_cfg if os.path.exists(P3_BEST) else colab_p2_cfg
print("Using checkpoint:", FINAL_CKPT)
!python -m src.export_onnx --config {FINAL_CFG} --checkpoint {FINAL_CKPT} --output {OUT_ONNX} --benchmark
print("ONNX files saved next to:", OUT_ONNX)

---
## Full Benchmark trên Tập `test` (647 bài, đối chứng 1-đối-1 với Model v4)

In [ ]:
FINAL_CKPT = P3_BEST if os.path.exists(P3_BEST) else P2_BEST
# Dùng cửa sổ đánh giá chuẩn của v5 để chấm toàn bộ 647 bài test
!python -m src.evaluate --config {colab_p2_cfg} --checkpoint {FINAL_CKPT} --split test --output {OUT_EVAL_JSON}

---
## Quick Inference Test (Đã tự ẩn các đoạn `N` không có âm thanh)

In [ ]:
FINAL_CKPT = P3_BEST if os.path.exists(P3_BEST) else P2_BEST
test_audio_dir = os.path.join(DATA_DST, "test", "audio")
test_files = sorted(os.listdir(test_audio_dir))[:3]
for f in test_files:
    audio_path = os.path.join(test_audio_dir, f)
    !python -m src.infer --config {colab_p2_cfg} --checkpoint {FINAL_CKPT} --audio {audio_path}